In [ ]:
import numpy as np
import pandas as pd

# Load UCI HAR training data
X_train = np.loadtxt("X_train.txt")
y_train = np.loadtxt("y_train.txt").astype(int)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nFirst 5 labels:")
print(y_train[:5])

print("\nUnique labels:")
print(np.unique(y_train))

X_train shape: (7352, 561)
y_train shape: (7352,)

First 5 labels:
[5 5 5 5 5]

Unique labels:
[1 2 3 4 5 6]


In [ ]:
activity_names = {
    1: "WALKING",
    2: "WALKING_UPSTAIRS",
    3: "WALKING_DOWNSTAIRS",
    4: "SITTING",
    5: "STANDING",
    6: "LAYING"
}

print("First 10 ground-truth activities:\n")

for i in range(10):
    print(
        f"Sample {i+1}: "
        f"Label = {y_train[i]}, "
        f"Activity = {activity_names[y_train[i]]}"
    )

First 10 ground-truth activities:

Sample 1: Label = 5, Activity = STANDING
Sample 2: Label = 5, Activity = STANDING
Sample 3: Label = 5, Activity = STANDING
Sample 4: Label = 5, Activity = STANDING
Sample 5: Label = 5, Activity = STANDING
Sample 6: Label = 5, Activity = STANDING
Sample 7: Label = 5, Activity = STANDING
Sample 8: Label = 5, Activity = STANDING
Sample 9: Label = 5, Activity = STANDING
Sample 10: Label = 5, Activity = STANDING


In [ ]:
# Count samples belonging to each activity
unique, counts = np.unique(y_train, return_counts=True)

print("Training class distribution:\n")

for label, count in zip(unique, counts):
    percentage = (count / len(y_train)) * 100

    print(
        f"{label} - {activity_names[label]:20s}: "
        f"{count:4d} samples ({percentage:.2f}%)"
    )


Training class distribution:

1 - WALKING             : 1226 samples (16.68%)
2 - WALKING_UPSTAIRS    : 1073 samples (14.59%)
3 - WALKING_DOWNSTAIRS  :  986 samples (13.41%)
4 - SITTING             : 1286 samples (17.49%)
5 - STANDING            : 1374 samples (18.69%)
6 - LAYING              : 1407 samples (19.14%)


In [ ]:
# Load test data
X_test = np.loadtxt("X_test.txt")
y_test = np.loadtxt("y_test.txt").astype(int)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nFirst 5 test labels:")
print(y_test[:5])

print("\nUnique test labels:")
print(np.unique(y_test))

X_test shape: (2947, 561)
y_test shape: (2947,)

First 5 test labels:
[5 5 5 5 5]

Unique test labels:
[1 2 3 4 5 6]


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Convert labels from 1-6 to 0-5 for TensorFlow
y_train_nn = y_train - 1
y_test_nn = y_test - 1

# Reference classifier
reference_model = Sequential([
    tf.keras.Input(shape=(561,)),

    Dense(256, activation="relu"),
    Dropout(0.3),

    Dense(128, activation="relu"),
    Dropout(0.2),

    Dense(64, activation="relu"),

    Dense(6, activation="softmax")
])

reference_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

reference_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       143,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 185,414 (724.27 KB)

 Trainable params: 185,414 (724.27 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = reference_model.fit(
    X_train,
    y_train_nn,

    epochs=30,
    batch_size=64,

    validation_split=0.2,

    verbose=1
)

Epoch 1/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.6800 - loss: 0.7370 - val_accuracy: 0.8933 - val_loss: 0.2588
Epoch 2/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8766 - loss: 0.2907 - val_accuracy: 0.9293 - val_loss: 0.1590
Epoch 3/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9169 - loss: 0.1980 - val_accuracy: 0.9327 - val_loss: 0.1619
Epoch 4/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9378 - loss: 0.1499 - val_accuracy: 0.9293 - val_loss: 0.1657
Epoch 5/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9478 - loss: 0.1320 - val_accuracy: 0.9429 - val_loss: 0.1507
Epoch 6/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9514 - loss: 0.1230 - val_accuracy: 0.9381 - val_loss: 0.1544
Epoch 7/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9607 - loss: 0.1014 - val_accuracy: 0.9456 - val_loss: 0.1457
Epoch 8/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9515 - loss: 0.1250 - val_accuracy: 0.9409 - val_loss

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Predict probabilities for the test samples
y_prob = reference_model.predict(X_test)

# Convert probabilities into predicted class (0-5)
y_pred_nn = np.argmax(y_prob, axis=1)

# Convert back to original HAR labels (1-6)
y_pred = y_pred_nn + 1

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred,
    average="macro"
)

recall = recall_score(
    y_test, y_pred,
    average="macro"
)

f1 = f1_score(
    y_test, y_pred,
    average="macro"
)

print("REFERENCE CLASSIFIER - TEST RESULTS")
print("-----------------------------------")
print(f"Accuracy        : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Macro Precision : {precision:.4f}")
print(f"Macro Recall    : {recall:.4f}")
print(f"Macro F1-score  : {f1:.4f}")

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
REFERENCE CLASSIFIER - TEST RESULTS
-----------------------------------
Accuracy        : 0.9447 (94.47%)
Macro Precision : 0.9466
Macro Recall    : 0.9445
Macro F1-score  : 0.9449


In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense

# Semantic latent dimension
latent_dim = 32

# Input: 561 HAR features
semantic_input = Input(shape=(561,), name="HAR_Input")

# Semantic encoder
x = Dense(256, activation="relu")(semantic_input)
x = Dense(128, activation="relu")(x)

semantic_vector = Dense(
    latent_dim,
    activation="linear",
    name="Semantic_Vector"
)(x)

semantic_encoder = Model(
    inputs=semantic_input,
    outputs=semantic_vector,
    name="Semantic_Encoder"
)

semantic_encoder.summary()

Model: "Semantic_Encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ HAR_Input (InputLayer)          │ (None, 561)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       143,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Semantic_Vector (Dense)         │ (None, 32)             │         4,128 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 180,896 (706.62 KB)

 Trainable params: 180,896 (706.62 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
class AWGNChannel(tf.keras.layers.Layer):

    def __init__(self, snr_db=10.0, **kwargs):
        super().__init__(**kwargs)
        self.snr_db = snr_db

    def call(self, z):

        # --------------------------------
        # 1. Normalize transmit power
        # --------------------------------

        power = tf.reduce_mean(
            tf.square(z),
            axis=1,
            keepdims=True
        )

        z_normalized = z / tf.sqrt(power + 1e-8)

        # --------------------------------
        # 2. Convert SNR dB to noise power
        # --------------------------------

        snr_linear = 10.0 ** (self.snr_db / 10.0)

        noise_variance = 1.0 / snr_linear

        noise_std = tf.sqrt(noise_variance)

        # --------------------------------
        # 3. Generate Gaussian noise
        # --------------------------------

        noise = tf.random.normal(
            shape=tf.shape(z_normalized),
            mean=0.0,
            stddev=noise_std
        )

        # --------------------------------
        # 4. Received semantic vector
        # --------------------------------

        z_received = z_normalized + noise

        return z_received

In [ ]:
# ==============================
# TASK DECODER
# ==============================

decoder_input = Input(
    shape=(latent_dim,),
    name="Received_Semantic_Vector"
)

x = Dense(64, activation="relu")(decoder_input)
x = Dense(32, activation="relu")(x)

activity_output = Dense(
    6,
    activation="softmax",
    name="Activity_Prediction"
)(x)

task_decoder = Model(
    inputs=decoder_input,
    outputs=activity_output,
    name="Task_Decoder"
)

task_decoder.summary()

Model: "Task_Decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Received_Semantic_Vector        │ (None, 32)             │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Activity_Prediction (Dense)     │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,390 (17.15 KB)

 Trainable params: 4,390 (17.15 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ==========================================
# COMPLETE SEMANTIC COMMUNICATION MODEL
# ==========================================

# Original HAR input
system_input = Input(
    shape=(561,),
    name="Original_HAR_Features"
)

# 1. Semantic encoding
z = semantic_encoder(system_input)

# 2. AWGN channel at 10 dB
channel = AWGNChannel(
    snr_db=10.0,
    name="AWGN_Channel"
)

z_received = channel(z)

# 3. Receiver-side task decoding
system_output = task_decoder(z_received)

# Complete end-to-end model
semantic_comm_model = Model(
    inputs=system_input,
    outputs=system_output,
    name="Task_Oriented_Semantic_Communication"
)

# Compile
semantic_comm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

semantic_comm_model.summary()

Model: "Task_Oriented_Semantic_Communication"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Original_HAR_Features           │ (None, 561)            │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Semantic_Encoder (Functional)   │ (None, 32)             │       180,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ AWGN_Channel (AWGNChannel)      │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Task_Decoder (Functional)       │ (None, 6)              │         4,390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 185,286 (723.77 KB)

 Trainable params: 185,286 (723.77 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_semantic = semantic_comm_model.fit(
    X_train,
    y_train_nn,

    epochs=30,
    batch_size=64,

    validation_split=0.2,

    verbose=1
)

Epoch 1/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7465 - loss: 0.6440 - val_accuracy: 0.9313 - val_loss: 0.1976
Epoch 2/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9274 - loss: 0.1884 - val_accuracy: 0.9184 - val_loss: 0.1964
Epoch 3/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9475 - loss: 0.1366 - val_accuracy: 0.9361 - val_loss: 0.1622
Epoch 4/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9646 - loss: 0.0937 - val_accuracy: 0.9320 - val_loss: 0.1825
Epoch 5/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9565 - loss: 0.1081 - val_accuracy: 0.9273 - val_loss: 0.1863
Epoch 6/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9575 - loss: 0.1041 - val_accuracy: 0.9490 - val_loss: 0.1382
Epoch 7/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9750 - loss: 0.0644 - val_accuracy: 0.9422 - val_loss: 0.1453
Epoch 8/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9757 - loss: 0.0673 - val_accuracy: 0.9381 - val_loss

In [ ]:
# ==========================================
# TEST SEMANTIC COMMUNICATION MODEL AT 10 dB
# ==========================================

# Predict activity probabilities
y_prob_semantic = semantic_comm_model.predict(X_test)

# Predicted classes: 0-5
y_pred_semantic_nn = np.argmax(
    y_prob_semantic,
    axis=1
)

# Convert back to original HAR labels: 1-6
y_pred_semantic = y_pred_semantic_nn + 1

# Calculate metrics
semantic_accuracy = accuracy_score(
    y_test,
    y_pred_semantic
)

semantic_precision = precision_score(
    y_test,
    y_pred_semantic,
    average="macro"
)

semantic_recall = recall_score(
    y_test,
    y_pred_semantic,
    average="macro"
)

semantic_f1 = f1_score(
    y_test,
    y_pred_semantic,
    average="macro"
)

print("SEMANTIC COMMUNICATION - TEST RESULTS")
print("SNR = 10 dB")
print("--------------------------------------")

print(
    f"Accuracy        : {semantic_accuracy:.4f} "
    f"({semantic_accuracy*100:.2f}%)"
)

print(f"Macro Precision : {semantic_precision:.4f}")
print(f"Macro Recall    : {semantic_recall:.4f}")
print(f"Macro F1-score  : {semantic_f1:.4f}")

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
SEMANTIC COMMUNICATION - TEST RESULTS
SNR = 10 dB
--------------------------------------
Accuracy        : 0.9372 (93.72%)
Macro Precision : 0.9413
Macro Recall    : 0.9358
Macro F1-score  : 0.9371


In [ ]:
# Get the AWGN channel from the trained model
awgn_layer = semantic_comm_model.get_layer("AWGN_Channel")

print("Current SNR:", awgn_layer.snr_db, "dB")

Current SNR: 10.0 dB


In [ ]:
awgn_layer.snr_db = 20.0

print("New SNR:", awgn_layer.snr_db, "dB")

New SNR: 20.0 dB


In [ ]:
# ==========================================
# TEST AT 20 dB
# ==========================================

# Make sure channel is at 20 dB
awgn_layer.snr_db = 20.0

# Prediction
y_prob_20 = semantic_comm_model.predict(X_test)

# Convert probabilities to original labels 1-6
y_pred_20 = np.argmax(y_prob_20, axis=1) + 1

# Metrics
accuracy_20 = accuracy_score(y_test, y_pred_20)

precision_20 = precision_score(
    y_test, y_pred_20, average="macro"
)

recall_20 = recall_score(
    y_test, y_pred_20, average="macro"
)

f1_20 = f1_score(
    y_test, y_pred_20, average="macro"
)

print("SEMANTIC COMMUNICATION - TEST RESULTS")
print("SNR = 20 dB")
print("--------------------------------------")
print(f"Accuracy        : {accuracy_20:.4f} ({accuracy_20*100:.2f}%)")
print(f"Macro Precision : {precision_20:.4f}")
print(f"Macro Recall    : {recall_20:.4f}")
print(f"Macro F1-score  : {f1_20:.4f}")

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
SEMANTIC COMMUNICATION - TEST RESULTS
SNR = 20 dB
--------------------------------------
Accuracy        : 0.9379 (93.79%)
Macro Precision : 0.9421
Macro Recall    : 0.9365
Macro F1-score  : 0.9379


In [ ]:
snr_values = [15, 10, 5, 0]

for snr in snr_values:

    # Change channel SNR
    awgn_layer.snr_db = float(snr)

    # Prediction
    y_prob_snr = semantic_comm_model.predict(
        X_test,
        verbose=0
    )

    # Convert to labels 1-6
    y_pred_snr = np.argmax(
        y_prob_snr,
        axis=1
    ) + 1

    # Metrics
    acc = accuracy_score(
        y_test,
        y_pred_snr
    )

    prec = precision_score(
        y_test,
        y_pred_snr,
        average="macro"
    )

    rec = recall_score(
        y_test,
        y_pred_snr,
        average="macro"
    )

    f1 = f1_score(
        y_test,
        y_pred_snr,
        average="macro"
    )

    print(f"SNR = {snr} dB")
    print(f"Accuracy        : {acc:.4f} ({acc*100:.2f}%)")
    print(f"Macro Precision : {prec:.4f}")
    print(f"Macro Recall    : {rec:.4f}")
    print(f"Macro F1-score  : {f1:.4f}")
    print("--------------------------------")

SNR = 15 dB
Accuracy        : 0.9359 (93.59%)
Macro Precision : 0.9401
Macro Recall    : 0.9344
Macro F1-score  : 0.9357
--------------------------------
SNR = 10 dB
Accuracy        : 0.9379 (93.79%)
Macro Precision : 0.9418
Macro Recall    : 0.9367
Macro F1-score  : 0.9379
--------------------------------
SNR = 5 dB
Accuracy        : 0.9379 (93.79%)
Macro Precision : 0.9421
Macro Recall    : 0.9365
Macro F1-score  : 0.9378
--------------------------------
SNR = 0 dB
Accuracy        : 0.9362 (93.62%)
Macro Precision : 0.9409
Macro Recall    : 0.9347
Macro F1-score  : 0.9362
--------------------------------


In [ ]:
snr_values_low = [0, -5, -10, -15, -20]

for snr in snr_values_low:

    awgn_layer.snr_db = float(snr)

    y_prob_snr = semantic_comm_model.predict(
        X_test,
        verbose=0
    )

    y_pred_snr = np.argmax(
        y_prob_snr,
        axis=1
    ) + 1

    acc = accuracy_score(y_test, y_pred_snr)

    f1 = f1_score(
        y_test,
        y_pred_snr,
        average="macro"
    )

    print(f"SNR = {snr:>3} dB | "
          f"Accuracy = {acc*100:.2f}% | "
          f"Macro F1 = {f1:.4f}")

SNR =   0 dB | Accuracy = 93.76% | Macro F1 = 0.9375
SNR =  -5 dB | Accuracy = 93.76% | Macro F1 = 0.9375
SNR = -10 dB | Accuracy = 93.55% | Macro F1 = 0.9357
SNR = -15 dB | Accuracy = 93.65% | Macro F1 = 0.9365
SNR = -20 dB | Accuracy = 93.76% | Macro F1 = 0.9375


In [ ]:
awgn_layer.snr_db = 10.0

In [ ]:
# Take 100 test samples
X_debug = X_test[:100]

# Generate semantic vectors WITHOUT channel
z_debug = semantic_encoder(X_debug, training=False)

# Normalize them exactly as the channel does
power = tf.reduce_mean(
    tf.square(z_debug),
    axis=1,
    keepdims=True
)

z_tx = z_debug / tf.sqrt(power + 1e-8)

print("Average normalized signal power:",
      tf.reduce_mean(tf.square(z_tx)).numpy())

Average normalized signal power: 0.99999994


In [ ]:
for snr in [20, 10, 0, -10, -20]:

    awgn_layer.snr_db = float(snr)

    z_rx = awgn_layer(z_debug)

    # Actual noise added
    noise = z_rx - z_tx

    signal_power = tf.reduce_mean(
        tf.square(z_tx)
    ).numpy()

    noise_power = tf.reduce_mean(
        tf.square(noise)
    ).numpy()

    measured_snr = 10 * np.log10(
        signal_power / noise_power
    )

    print(
        f"Set SNR = {snr:>3} dB | "
        f"Signal Power = {signal_power:.4f} | "
        f"Noise Power = {noise_power:.4f} | "
        f"Measured SNR = {measured_snr:.2f} dB"
    )

Set SNR =  20 dB | Signal Power = 1.0000 | Noise Power = 0.0098 | Measured SNR = 20.07 dB
Set SNR =  10 dB | Signal Power = 1.0000 | Noise Power = 0.0989 | Measured SNR = 10.05 dB
Set SNR =   0 dB | Signal Power = 1.0000 | Noise Power = 1.0061 | Measured SNR = -0.03 dB
Set SNR = -10 dB | Signal Power = 1.0000 | Noise Power = 10.0773 | Measured SNR = -10.03 dB
Set SNR = -20 dB | Signal Power = 1.0000 | Noise Power = 100.7819 | Measured SNR = -20.03 dB


In [ ]:
awgn_layer.snr_db = -20.0

# Step 1: Semantic encoding
z_test = semantic_encoder(
    X_test,
    training=False
)

# Step 2: Pass through the -20 dB channel
z_received_test = awgn_layer(
    z_test
)

# Step 3: Receiver task decoding
y_prob_manual = task_decoder(
    z_received_test,
    training=False
).numpy()

# Step 4: Predicted activity
y_pred_manual = np.argmax(
    y_prob_manual,
    axis=1
) + 1

# Step 5: Evaluate
acc_manual = accuracy_score(
    y_test,
    y_pred_manual
)

f1_manual = f1_score(
    y_test,
    y_pred_manual,
    average="macro"
)

print("MANUAL PIPELINE TEST")
print("SNR = -20 dB")
print("--------------------")
print(f"Accuracy : {acc_manual*100:.2f}%")
print(f"Macro F1 : {f1_manual:.4f}")

MANUAL PIPELINE TEST
SNR = -20 dB
--------------------
Accuracy : 27.08%
Macro F1 : 0.2680


In [ ]:
snr_values = [20, 15, 10, 5, 0, -5, -10, -15, -20]

results = []

# Encode test data once
z_test = semantic_encoder(
    X_test,
    training=False
)

for snr in snr_values:

    # Set channel condition
    awgn_layer.snr_db = float(snr)

    # Pass semantic representation through channel
    z_received = awgn_layer(z_test)

    # Receiver prediction
    y_prob = task_decoder(
        z_received,
        training=False
    ).numpy()

    # Convert to original labels 1-6
    y_pred = np.argmax(y_prob, axis=1) + 1

    # Metrics
    acc = accuracy_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    results.append([snr, acc, f1])

    print(
        f"SNR = {snr:>3} dB | "
        f"Accuracy = {acc*100:.2f}% | "
        f"Macro F1 = {f1:.4f}"
    )

# Reset channel to training SNR
awgn_layer.snr_db = 10.0

SNR =  20 dB | Accuracy = 93.55% | Macro F1 = 0.9354
SNR =  15 dB | Accuracy = 93.52% | Macro F1 = 0.9353
SNR =  10 dB | Accuracy = 93.79% | Macro F1 = 0.9377
SNR =   5 dB | Accuracy = 93.48% | Macro F1 = 0.9347
SNR =   0 dB | Accuracy = 92.64% | Macro F1 = 0.9262
SNR =  -5 dB | Accuracy = 82.90% | Macro F1 = 0.8305
SNR = -10 dB | Accuracy = 59.18% | Macro F1 = 0.5899
SNR = -15 dB | Accuracy = 39.67% | Macro F1 = 0.3923
SNR = -20 dB | Accuracy = 29.05% | Macro F1 = 0.2863


In [ ]:
import numpy as np

snr_values = [20, 15, 10, 5, 0, -5, -10, -15, -20]

num_runs = 20

# Encode X_test only once
z_test = semantic_encoder(
    X_test,
    training=False
)

final_results = []

for snr in snr_values:

    accuracies = []
    f1_scores = []

    awgn_layer.snr_db = float(snr)

    for run in range(num_runs):

        # New AWGN realization
        z_received = awgn_layer(z_test)

        # Receiver prediction
        y_prob = task_decoder(
            z_received,
            training=False
        ).numpy()

        y_pred = np.argmax(
            y_prob,
            axis=1
        ) + 1

        # Metrics
        acc = accuracy_score(
            y_test,
            y_pred
        )

        f1 = f1_score(
            y_test,
            y_pred,
            average="macro"
        )

        accuracies.append(acc)
        f1_scores.append(f1)

    # Mean and standard deviation
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)

    mean_f1 = np.mean(f1_scores)
    std_f1 = np.std(f1_scores)

    final_results.append([
        snr,
        mean_acc,
        std_acc,
        mean_f1,
        std_f1
    ])

    print(
        f"SNR = {snr:>3} dB | "
        f"Accuracy = {mean_acc*100:.2f}% ± {std_acc*100:.2f}% | "
        f"Macro F1 = {mean_f1:.4f} ± {std_f1:.4f}"
    )

# Restore training SNR
awgn_layer.snr_db = 10.0

SNR =  20 dB | Accuracy = 93.70% ± 0.10% | Macro F1 = 0.9370 ± 0.0010
SNR =  15 dB | Accuracy = 93.67% ± 0.11% | Macro F1 = 0.9366 ± 0.0011
SNR =  10 dB | Accuracy = 93.67% ± 0.11% | Macro F1 = 0.9366 ± 0.0011
SNR =   5 dB | Accuracy = 93.54% ± 0.16% | Macro F1 = 0.9354 ± 0.0016
SNR =   0 dB | Accuracy = 92.74% ± 0.26% | Macro F1 = 0.9275 ± 0.0025
SNR =  -5 dB | Accuracy = 83.17% ± 0.62% | Macro F1 = 0.8324 ± 0.0062
SNR = -10 dB | Accuracy = 58.85% ± 0.97% | Macro F1 = 0.5873 ± 0.0099
SNR = -15 dB | Accuracy = 39.20% ± 0.69% | Macro F1 = 0.3888 ± 0.0072
SNR = -20 dB | Accuracy = 28.40% ± 0.86% | Macro F1 = 0.2801 ± 0.0085


In [ ]:
# ==========================================
# RECEIVER-SIDE ENTROPY AT 10 dB
# ==========================================

awgn_layer.snr_db = 10.0

# 1. Encode test samples
z_test = semantic_encoder(
    X_test,
    training=False
)

# 2. Pass through channel
z_received_10 = awgn_layer(z_test)

# 3. Receiver probabilities
prob_10 = task_decoder(
    z_received_10,
    training=False
).numpy()

# 4. Normalized Shannon entropy
epsilon = 1e-12

entropy_10 = -np.sum(
    prob_10 * np.log2(prob_10 + epsilon),
    axis=1
)

normalized_entropy_10 = entropy_10 / np.log2(6)

# 5. Predicted labels
pred_10 = np.argmax(prob_10, axis=1) + 1

# Ground truth is used ONLY for experimental analysis
correct_10 = (pred_10 == y_test)

print("Number of test samples:", len(normalized_entropy_10))

print(
    "Mean entropy - CORRECT predictions:",
    normalized_entropy_10[correct_10].mean()
)

print(
    "Mean entropy - WRONG predictions:",
    normalized_entropy_10[~correct_10].mean()
)

print(
    "Number correct:",
    correct_10.sum()
)

print(
    "Number wrong:",
    (~correct_10).sum()
)

Number of test samples: 2947
Mean entropy - CORRECT predictions: 0.02004307273418228
Mean entropy - WRONG predictions: 0.14664901516255285
Number correct: 2759
Number wrong: 188


In [ ]:
# ==========================================
# ENTROPY DISTRIBUTION ANALYSIS
# ==========================================

correct_entropy = normalized_entropy_10[correct_10]
wrong_entropy = normalized_entropy_10[~correct_10]

percentiles = [0, 10, 25, 50, 75, 90, 95, 100]

print("CORRECT PREDICTIONS")
print("-------------------")

for p in percentiles:
    print(
        f"{p:>3}th percentile: "
        f"{np.percentile(correct_entropy, p):.4f}"
    )

print("\nWRONG PREDICTIONS")
print("-----------------")

for p in percentiles:
    print(
        f"{p:>3}th percentile: "
        f"{np.percentile(wrong_entropy, p):.4f}"
    )

CORRECT PREDICTIONS
-------------------
  0th percentile: 0.0000
 10th percentile: 0.0001
 25th percentile: 0.0003
 50th percentile: 0.0012
 75th percentile: 0.0057
 90th percentile: 0.0297
 95th percentile: 0.1394
100th percentile: 0.7155

WRONG PREDICTIONS
-----------------
  0th percentile: 0.0004
 10th percentile: 0.0038
 25th percentile: 0.0140
 50th percentile: 0.0761
 75th percentile: 0.2841
 90th percentile: 0.3695
 95th percentile: 0.3850
100th percentile: 0.4683


In [ ]:
# ==========================================
# ENTROPY VS SNR
# ==========================================

snr_values = [20, 15, 10, 5, 0, -5, -10, -15, -20]

num_runs = 20

entropy_results = []

# Encode once
z_test = semantic_encoder(
    X_test,
    training=False
)

for snr in snr_values:

    run_entropies = []

    awgn_layer.snr_db = float(snr)

    for run in range(num_runs):

        # Channel
        z_received = awgn_layer(z_test)

        # Decoder probabilities
        probabilities = task_decoder(
            z_received,
            training=False
        ).numpy()

        # Normalized entropy for every sample
        entropy = -np.sum(
            probabilities *
            np.log2(probabilities + 1e-12),
            axis=1
        )

        normalized_entropy = (
            entropy / np.log2(6)
        )

        # Mean entropy for this channel realization
        run_entropies.append(
            np.mean(normalized_entropy)
        )

    mean_entropy = np.mean(run_entropies)
    std_entropy = np.std(run_entropies)

    entropy_results.append([
        snr,
        mean_entropy,
        std_entropy
    ])

    print(
        f"SNR = {snr:>3} dB | "
        f"Mean Normalized Entropy = "
        f"{mean_entropy:.4f} ± {std_entropy:.4f}"
    )

# Restore
awgn_layer.snr_db = 10.0

SNR =  20 dB | Mean Normalized Entropy = 0.0290 ± 0.0002
SNR =  15 dB | Mean Normalized Entropy = 0.0292 ± 0.0003
SNR =  10 dB | Mean Normalized Entropy = 0.0293 ± 0.0004
SNR =   5 dB | Mean Normalized Entropy = 0.0311 ± 0.0006
SNR =   0 dB | Mean Normalized Entropy = 0.0472 ± 0.0013
SNR =  -5 dB | Mean Normalized Entropy = 0.1106 ± 0.0029
SNR = -10 dB | Mean Normalized Entropy = 0.1408 ± 0.0025
SNR = -15 dB | Mean Normalized Entropy = 0.1054 ± 0.0033
SNR = -20 dB | Mean Normalized Entropy = 0.0634 ± 0.0020


In [ ]:
# ==========================================
# RECEIVED LATENT ENERGY VS SNR
# ==========================================

snr_values = [20, 15, 10, 5, 0, -5, -10, -15, -20]

num_runs = 20

energy_results = []

z_test = semantic_encoder(
    X_test,
    training=False
)

for snr in snr_values:

    run_energies = []

    awgn_layer.snr_db = float(snr)

    for run in range(num_runs):

        # Received semantic vector
        z_received = awgn_layer(z_test)

        # Energy of each received vector
        sample_energy = tf.reduce_mean(
            tf.square(z_received),
            axis=1
        ).numpy()

        # Mean over all test samples
        run_energies.append(
            np.mean(sample_energy)
        )

    mean_energy = np.mean(run_energies)
    std_energy = np.std(run_energies)

    energy_results.append([
        snr,
        mean_energy,
        std_energy
    ])

    print(
        f"SNR = {snr:>3} dB | "
        f"Received Latent Energy = "
        f"{mean_energy:.4f} ± {std_energy:.4f}"
    )

# Restore
awgn_layer.snr_db = 10.0

SNR =  20 dB | Received Latent Energy = 1.0101 ± 0.0006
SNR =  15 dB | Received Latent Energy = 1.0316 ± 0.0009
SNR =  10 dB | Received Latent Energy = 1.1000 ± 0.0018
SNR =   5 dB | Received Latent Energy = 1.3157 ± 0.0046
SNR =   0 dB | Received Latent Energy = 2.0013 ± 0.0079
SNR =  -5 dB | Received Latent Energy = 4.1626 ± 0.0202
SNR = -10 dB | Received Latent Energy = 11.0034 ± 0.0493
SNR = -15 dB | Received Latent Energy = 32.5836 ± 0.1388
SNR = -20 dB | Received Latent Energy = 100.9848 ± 0.4693


In [ ]:
# ==========================================
# CLEAN LATENT-SPACE CLASS CENTROIDS
# ==========================================

# Get clean semantic vectors
z_clean = semantic_encoder(
    X_test,
    training=False
).numpy()

print("Latent matrix shape:", z_clean.shape)

activity_names = {
    1: "WALKING",
    2: "WALKING_UPSTAIRS",
    3: "WALKING_DOWNSTAIRS",
    4: "SITTING",
    5: "STANDING",
    6: "LAYING"
}

class_centroids = {}

for class_id in range(1, 7):

    # Semantic vectors belonging to this activity
    class_vectors = z_clean[
        y_test == class_id
    ]

    # Mean 32-D semantic vector
    centroid = np.mean(
        class_vectors,
        axis=0
    )

    class_centroids[class_id] = centroid

    print(
        f"{class_id}: "
        f"{activity_names[class_id]:20s} | "
        f"Samples = {len(class_vectors):4d} | "
        f"Centroid norm = {np.linalg.norm(centroid):.4f}"
    )

Latent matrix shape: (2947, 32)
1: WALKING              | Samples =  496 | Centroid norm = 4.2819
2: WALKING_UPSTAIRS     | Samples =  471 | Centroid norm = 4.7532
3: WALKING_DOWNSTAIRS   | Samples =  420 | Centroid norm = 6.8301
4: SITTING              | Samples =  491 | Centroid norm = 3.3500
5: STANDING             | Samples =  532 | Centroid norm = 4.1942
6: LAYING               | Samples =  537 | Centroid norm = 7.6857


In [ ]:
# ==========================================
# DISTANCE BETWEEN CLASS CENTROIDS
# ==========================================

centroid_distance_matrix = np.zeros((6, 6))

for i in range(1, 7):
    for j in range(1, 7):

        centroid_distance_matrix[i-1, j-1] = np.linalg.norm(
            class_centroids[i] -
            class_centroids[j]
        )

print("CENTROID DISTANCE MATRIX")
print("------------------------")

print(
    "          ",
    "   ".join([f"C{i}" for i in range(1, 7)])
)

for i in range(6):

    row = "   ".join(
        [f"{centroid_distance_matrix[i,j]:.3f}"
         for j in range(6)]
    )

    print(f"C{i+1}     {row}")


print("\nClass Mapping:")
for class_id in range(1, 7):
    print(
        f"C{class_id} = "
        f"{activity_names[class_id]}"
    )

CENTROID DISTANCE MATRIX
------------------------
           C1   C2   C3   C4   C5   C6
C1     0.000   6.659   8.354   6.816   6.170   8.535
C2     6.659   0.000   9.107   5.826   5.201   9.036
C3     8.354   9.107   0.000   8.698   8.426   11.195
C4     6.816   5.826   8.698   0.000   5.683   8.106
C5     6.170   5.201   8.426   5.683   0.000   9.297
C6     8.535   9.036   11.195   8.106   9.297   0.000

Class Mapping:
C1 = WALKING
C2 = WALKING_UPSTAIRS
C3 = WALKING_DOWNSTAIRS
C4 = SITTING
C5 = STANDING
C6 = LAYING


In [ ]:
# ==========================================
# WITHIN-CLASS LATENT SPREAD
# ==========================================

class_spread = {}

for class_id in range(1, 7):

    # Clean latent vectors for this class
    class_vectors = z_clean[
        y_test == class_id
    ]

    # Corresponding centroid
    centroid = class_centroids[
        class_id
    ]

    # Euclidean distance of every sample
    # from its own class centroid
    distances = np.linalg.norm(
        class_vectors - centroid,
        axis=1
    )

    mean_distance = np.mean(distances)
    std_distance = np.std(distances)

    median_distance = np.median(distances)

    p95_distance = np.percentile(
        distances,
        95
    )

    class_spread[class_id] = {
        "mean": mean_distance,
        "std": std_distance,
        "median": median_distance,
        "p95": p95_distance
    }

    print(
        f"{class_id}: "
        f"{activity_names[class_id]:20s} | "
        f"Mean = {mean_distance:.4f} | "
        f"Std = {std_distance:.4f} | "
        f"Median = {median_distance:.4f} | "
        f"95th percentile = {p95_distance:.4f}"
    )

1: WALKING              | Mean = 1.7279 | Std = 1.0287 | Median = 1.4521 | 95th percentile = 3.6807
2: WALKING_UPSTAIRS     | Mean = 2.1022 | Std = 1.0529 | Median = 1.8743 | 95th percentile = 4.0722
3: WALKING_DOWNSTAIRS   | Mean = 3.9059 | Std = 2.0965 | Median = 3.2561 | 95th percentile = 7.6777
4: SITTING              | Mean = 2.0802 | Std = 1.4004 | Median = 1.7083 | 95th percentile = 4.9805
5: STANDING             | Mean = 1.6021 | Std = 0.9401 | Median = 1.3914 | 95th percentile = 3.4806
6: LAYING               | Mean = 2.7508 | Std = 1.8770 | Median = 2.2947 | 95th percentile = 7.9254


In [ ]:
# ==========================================
# BUILD SEMANTIC REFERENCE FROM TRAINING DATA
# ==========================================

# Clean latent vectors from TRAINING data
z_train_clean = semantic_encoder(
    X_train,
    training=False
).numpy()

train_centroids = {}
train_spread = {}

for class_id in range(1, 7):

    # Training vectors belonging to this class
    class_vectors = z_train_clean[
        y_train == class_id
    ]

    # Class centroid
    centroid = np.mean(
        class_vectors,
        axis=0
    )

    # Distance of each training sample
    # from its own centroid
    distances = np.linalg.norm(
        class_vectors - centroid,
        axis=1
    )

    train_centroids[class_id] = centroid

    train_spread[class_id] = {
        "mean": np.mean(distances),
        "std": np.std(distances),
        "median": np.median(distances),
        "p95": np.percentile(distances, 95)
    }

    print(
        f"{class_id}: "
        f"{activity_names[class_id]:20s} | "
        f"Samples = {len(class_vectors):4d} | "
        f"Mean distance = {np.mean(distances):.4f} | "
        f"Std = {np.std(distances):.4f} | "
        f"95th percentile = {np.percentile(distances, 95):.4f}"
    )

1: WALKING              | Samples = 1226 | Mean distance = 1.4125 | Std = 0.6269 | 95th percentile = 2.5877
2: WALKING_UPSTAIRS     | Samples = 1073 | Mean distance = 1.7897 | Std = 0.8274 | 95th percentile = 3.3985
3: WALKING_DOWNSTAIRS   | Samples =  986 | Mean distance = 3.6685 | Std = 2.1006 | 95th percentile = 7.3642
4: SITTING              | Samples = 1286 | Mean distance = 1.6776 | Std = 0.9755 | 95th percentile = 3.4453
5: STANDING             | Samples = 1374 | Mean distance = 1.4658 | Std = 0.8446 | 95th percentile = 3.0475
6: LAYING               | Samples = 1407 | Mean distance = 2.3132 | Std = 1.3672 | 95th percentile = 5.3963


In [ ]:
# ==========================================
# SEMANTIC-SPACE CONSISTENCY VS SNR
# ==========================================

snr_values = [20, 15, 10, 5, 0, -5, -10, -15, -20]

num_runs = 20

consistency_results = []

# Encode test data once
z_test = semantic_encoder(
    X_test,
    training=False
)

for snr in snr_values:

    run_mean_distances = []
    run_outside_rates = []

    awgn_layer.snr_db = float(snr)

    for run in range(num_runs):

        # -----------------------------
        # Receiver gets noisy latent
        # -----------------------------
        z_received = awgn_layer(
            z_test
        ).numpy()

        # -----------------------------
        # Task decoder prediction
        # -----------------------------
        probabilities = task_decoder(
            z_received,
            training=False
        ).numpy()

        predicted_classes = (
            np.argmax(probabilities, axis=1) + 1
        )

        normalized_distances = []

        # -----------------------------
        # Compare each received vector
        # with predicted-class region
        # -----------------------------
        for i in range(len(z_received)):

            predicted_class = predicted_classes[i]

            centroid = train_centroids[
                predicted_class
            ]

            radius_95 = train_spread[
                predicted_class
            ]["p95"]

            distance = np.linalg.norm(
                z_received[i] - centroid
            )

            normalized_distance = (
                distance / radius_95
            )

            normalized_distances.append(
                normalized_distance
            )

        normalized_distances = np.array(
            normalized_distances
        )

        # Mean normalized distance
        run_mean_distances.append(
            np.mean(normalized_distances)
        )

        # Fraction outside training 95% radius
        outside_rate = np.mean(
            normalized_distances > 1.0
        )

        run_outside_rates.append(
            outside_rate
        )

    # Average over 20 channel realizations
    mean_distance = np.mean(
        run_mean_distances
    )

    std_distance = np.std(
        run_mean_distances
    )

    mean_outside = np.mean(
        run_outside_rates
    )

    std_outside = np.std(
        run_outside_rates
    )

    consistency_results.append([
        snr,
        mean_distance,
        std_distance,
        mean_outside,
        std_outside
    ])

    print(
        f"SNR = {snr:>3} dB | "
        f"Normalized Distance = "
        f"{mean_distance:.3f} ± {std_distance:.3f} | "
        f"Outside Region = "
        f"{mean_outside*100:.2f}% ± "
        f"{std_outside*100:.2f}%"
    )

# Restore
awgn_layer.snr_db = 10.0

SNR =  20 dB | Normalized Distance = 0.668 ± 0.001 | Outside Region = 11.91% ± 0.12%
SNR =  15 dB | Normalized Distance = 0.711 ± 0.001 | Outside Region = 12.80% ± 0.19%
SNR =  10 dB | Normalized Distance = 0.828 ± 0.001 | Outside Region = 18.86% ± 0.34%
SNR =   5 dB | Normalized Distance = 1.108 ± 0.002 | Outside Region = 61.02% ± 0.59%
SNR =   0 dB | Normalized Distance = 1.699 ± 0.003 | Outside Region = 89.50% ± 0.43%
SNR =  -5 dB | Normalized Distance = 2.859 ± 0.009 | Outside Region = 99.95% ± 0.03%
SNR = -10 dB | Normalized Distance = 4.961 ± 0.030 | Outside Region = 100.00% ± 0.00%
SNR = -15 dB | Normalized Distance = 8.706 ± 0.059 | Outside Region = 100.00% ± 0.00%
SNR = -20 dB | Normalized Distance = 15.415 ± 0.085 | Outside Region = 100.00% ± 0.00%


In [ ]:
# ==========================================
# BUILD NORMALIZED TRANSMITTED-SPACE REFERENCE
# ==========================================

# Raw encoder outputs from training data
z_train_raw = semantic_encoder(
    X_train,
    training=False
)

# Apply EXACTLY the same normalization
# used before AWGN transmission
train_power = tf.reduce_mean(
    tf.square(z_train_raw),
    axis=1,
    keepdims=True
)

z_train_tx = (
    z_train_raw /
    tf.sqrt(train_power + 1e-8)
).numpy()

print(
    "Average transmitted training power:",
    np.mean(z_train_tx ** 2)
)

# ------------------------------------------
# Build class reference regions
# ------------------------------------------

tx_centroids = {}
tx_spread = {}

for class_id in range(1, 7):

    class_vectors = z_train_tx[
        y_train == class_id
    ]

    centroid = np.mean(
        class_vectors,
        axis=0
    )

    distances = np.linalg.norm(
        class_vectors - centroid,
        axis=1
    )

    tx_centroids[class_id] = centroid

    tx_spread[class_id] = {
        "mean": np.mean(distances),
        "std": np.std(distances),
        "median": np.median(distances),
        "p95": np.percentile(distances, 95)
    }

    print(
        f"{class_id}: "
        f"{activity_names[class_id]:20s} | "
        f"Mean = {np.mean(distances):.4f} | "
        f"Std = {np.std(distances):.4f} | "
        f"95th percentile = "
        f"{np.percentile(distances, 95):.4f}"
    )

Average transmitted training power: 1.0
1: WALKING              | Mean = 1.2307 | Std = 0.6125 | 95th percentile = 2.5902
2: WALKING_UPSTAIRS     | Mean = 1.2244 | Std = 0.5381 | 95th percentile = 2.2206
3: WALKING_DOWNSTAIRS   | Mean = 1.3391 | Std = 0.6970 | 95th percentile = 2.7596
4: SITTING              | Mean = 1.9261 | Std = 1.2517 | 95th percentile = 4.7280
5: STANDING             | Mean = 1.5872 | Std = 1.1660 | 95th percentile = 4.0000
6: LAYING               | Mean = 1.3335 | Std = 0.9837 | 95th percentile = 3.7638


In [ ]:
# ==========================================
# CORRECTED SEMANTIC-SPACE CONSISTENCY VS SNR
# ==========================================

snr_values = [20, 15, 10, 5, 0, -5, -10, -15, -20]

num_runs = 20

corrected_consistency_results = []

# Encode test data once
z_test = semantic_encoder(
    X_test,
    training=False
)

for snr in snr_values:

    run_mean_distances = []
    run_outside_rates = []

    awgn_layer.snr_db = float(snr)

    for run in range(num_runs):

        # Received semantic vector
        z_received = awgn_layer(
            z_test
        ).numpy()

        # Decoder probabilities
        probabilities = task_decoder(
            z_received,
            training=False
        ).numpy()

        # Predicted class
        predicted_classes = (
            np.argmax(probabilities, axis=1) + 1
        )

        normalized_distances = []

        for i in range(len(z_received)):

            predicted_class = predicted_classes[i]

            # CORRECTED transmitted-space centroid
            centroid = tx_centroids[
                predicted_class
            ]

            # CORRECTED transmitted-space 95% radius
            radius_95 = tx_spread[
                predicted_class
            ]["p95"]

            # Distance between received vector
            # and predicted semantic region
            distance = np.linalg.norm(
                z_received[i] - centroid
            )

            normalized_distance = (
                distance / radius_95
            )

            normalized_distances.append(
                normalized_distance
            )

        normalized_distances = np.array(
            normalized_distances
        )

        # Mean normalized distance
        run_mean_distances.append(
            np.mean(normalized_distances)
        )

        # Percentage outside reference region
        run_outside_rates.append(
            np.mean(normalized_distances > 1.0)
        )

    mean_distance = np.mean(
        run_mean_distances
    )

    std_distance = np.std(
        run_mean_distances
    )

    mean_outside = np.mean(
        run_outside_rates
    )

    std_outside = np.std(
        run_outside_rates
    )

    corrected_consistency_results.append([
        snr,
        mean_distance,
        std_distance,
        mean_outside,
        std_outside
    ])

    print(
        f"SNR = {snr:>3} dB | "
        f"Normalized Distance = "
        f"{mean_distance:.3f} ± {std_distance:.3f} | "
        f"Outside Region = "
        f"{mean_outside*100:.2f}% ± "
        f"{std_outside*100:.2f}%"
    )

# Restore
awgn_layer.snr_db = 10.0

SNR =  20 dB | Normalized Distance = 0.586 ± 0.001 | Outside Region = 12.62% ± 0.15%
SNR =  15 dB | Normalized Distance = 0.654 ± 0.001 | Outside Region = 13.91% ± 0.27%
SNR =  10 dB | Normalized Distance = 0.818 ± 0.002 | Outside Region = 22.11% ± 0.36%
SNR =   5 dB | Normalized Distance = 1.179 ± 0.003 | Outside Region = 57.45% ± 0.55%
SNR =   0 dB | Normalized Distance = 1.895 ± 0.004 | Outside Region = 99.09% ± 0.16%
SNR =  -5 dB | Normalized Distance = 3.236 ± 0.011 | Outside Region = 100.00% ± 0.00%
SNR = -10 dB | Normalized Distance = 5.581 ± 0.026 | Outside Region = 100.00% ± 0.00%
SNR = -15 dB | Normalized Distance = 9.809 ± 0.061 | Outside Region = 100.00% ± 0.00%
SNR = -20 dB | Normalized Distance = 17.375 ± 0.109 | Outside Region = 100.00% ± 0.00%


In [ ]:
# ==========================================
# SAMPLE-LEVEL ANALYSIS AT -10 dB
# ==========================================

awgn_layer.snr_db = -10.0

# Encode
z_test = semantic_encoder(
    X_test,
    training=False
)

# Channel
z_received = awgn_layer(
    z_test
).numpy()

# Decoder
probabilities = task_decoder(
    z_received,
    training=False
).numpy()

predicted_classes = (
    np.argmax(probabilities, axis=1) + 1
)

# ------------------------------------------
# Normalized semantic-space distance
# ------------------------------------------

normalized_distances = []

for i in range(len(z_received)):

    predicted_class = predicted_classes[i]

    centroid = tx_centroids[
        predicted_class
    ]

    radius_95 = tx_spread[
        predicted_class
    ]["p95"]

    distance = np.linalg.norm(
        z_received[i] - centroid
    )

    normalized_distances.append(
        distance / radius_95
    )

normalized_distances = np.array(
    normalized_distances
)

# ------------------------------------------
# Experimental ground truth
# NOT available to deployed receiver
# ------------------------------------------

correct_mask = (
    predicted_classes == y_test
)

correct_distances = normalized_distances[
    correct_mask
]

wrong_distances = normalized_distances[
    ~correct_mask
]

print("SNR = -10 dB")
print("-----------------------------")

print(
    "Correct predictions:",
    len(correct_distances)
)

print(
    "Wrong predictions:",
    len(wrong_distances)
)

print()

print(
    "Mean distance - CORRECT:",
    np.mean(correct_distances)
)

print(
    "Mean distance - WRONG:",
    np.mean(wrong_distances)
)

print()

print(
    "Median distance - CORRECT:",
    np.median(correct_distances)
)

print(
    "Median distance - WRONG:",
    np.median(wrong_distances)
)

SNR = -10 dB
-----------------------------
Correct predictions: 1707
Wrong predictions: 1240

Mean distance - CORRECT: 5.510063
Mean distance - WRONG: 5.5655594

Median distance - CORRECT: 5.1151123
Median distance - WRONG: 5.1925197


In [ ]:
# ==========================================
# CONFIDENTLY-WRONG ANALYSIS AT -10 dB
# ==========================================

# Normalized entropy for each sample
entropy = -np.sum(
    probabilities * np.log2(probabilities + 1e-12),
    axis=1
)

normalized_entropy = entropy / np.log2(6)

# Wrong predictions
wrong_mask = (
    predicted_classes != y_test
)

wrong_entropy = normalized_entropy[
    wrong_mask
]

wrong_distance = normalized_distances[
    wrong_mask
]

# Define "confidently wrong" using a strict
# receiver uncertainty threshold
confident_wrong_mask = (
    wrong_entropy < 0.05
)

confident_wrong_entropy = wrong_entropy[
    confident_wrong_mask
]

confident_wrong_distance = wrong_distance[
    confident_wrong_mask
]

print("Total wrong predictions:",
      len(wrong_entropy))

print("Confidently wrong (entropy < 0.05):",
      len(confident_wrong_entropy))

print(
    "Percentage of wrong predictions that are confidently wrong:",
    100 * len(confident_wrong_entropy) / len(wrong_entropy)
)

print()

print(
    "Mean entropy of confidently wrong:",
    np.mean(confident_wrong_entropy)
)

print(
    "Mean semantic distance of confidently wrong:",
    np.mean(confident_wrong_distance)
)

print(
    "Median semantic distance of confidently wrong:",
    np.median(confident_wrong_distance)
)

Total wrong predictions: 1240
Confidently wrong (entropy < 0.05): 400
Percentage of wrong predictions that are confidently wrong: 32.25806451612903

Mean entropy of confidently wrong: 0.013006131779664967
Mean semantic distance of confidently wrong: 5.6057644
Median semantic distance of confidently wrong: 5.237404
